# 10 - SHAP and explanation quality

Computing SHAP values is the easy half. The contribution is measuring whether the explanations are faithful, stable, and sparse - most XAI-in-security papers stop at a beeswarm plot.

In [ ]:
# --- standard header: every notebook starts with exactly this ---
from google.colab import drive; drive.mount('/content/drive')

REPO = '/content/secure-dns-trust-ai'
!git -C {REPO} pull -q 2>/dev/null || git clone -q https://github.com/sandesh20lamichhane/secure-dns-trust-ai.git {REPO}

import sys, os; sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'
%load_ext autoreload
%autoreload 2

from src.utils import config, manifest, seeds, io
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
import pandas as pd, xgboost as xgb
from src.explain import shap_tree, evaluate as xeval
from src.evaluate import splits
from src.features.build import to_matrix

RUN = 'fusion_family_disjoint_v1_s42_0004'
model = xgb.XGBClassifier(); model.load_model(f"{P['artifacts']['models']}/{RUN}.json")
df = pd.read_parquet(f"{P['data']['features']}/fused_v1.parquet")
split = splits.load_split(P['data']['splits'], 'family_disjoint_v1')
_, _, te = splits.apply_split(df, split)
Xte, yte, names = to_matrix(te)

In [ ]:
values, Xs, explainer, path = shap_tree.compute(model, Xte, RUN, P['artifacts']['shap_values'])
imp = shap_tree.global_importance(values, Xs.columns)
imp.head(25)

In [ ]:
fid = xeval.fidelity(model, Xs, values, k=5)
stab = xeval.stability(values, Xs)
spars = xeval.sparsity(values)
print(fid); print(stab); print(spars)
# fidelity_ratio >> 1 means top-SHAP features genuinely drive the prediction

In [ ]:
pd.DataFrame([{**fid, **stab, **spars, 'run_id': RUN}]).to_csv(
    f"{P['results']['tables']}/explanation_quality_{RUN}.csv", index=False)